# ESCM 2.0 — Work package 1: isolate the original mechanisms

**Question.** What do self-excitation, lateral inhibition, and the persistence gate actually do in the existing ESCM, before we replace any of them?

We use the same model equations, corrected neuron-local counter reset, custom-event scheduling, and explicitly ordered recurrent pathways as `01_building_escm.ipynb`. This is **ESCM 1.x baseline behavior**, not an ESCM 2.0 redesign.

The experiments separate four causal conditions: single-pixel stimulation, repeated stimulation, successive neighboring stimulation, and simultaneous neighboring stimulation. We use a **connected 5 × 5 network**, not a one-neuron network, because the current code implements self-excitation via the source side of an **outgoing recurrent synapse**. In a one-neuron network with no outgoing synapses, that self-excitation mechanism would disappear, invalidating the comparison.

Parameters default to the current legacy-script configuration. The paper's reported threshold and input amplification differ; those should be investigated separately rather than silently mixed into this baseline.

## 1. Define a controlled input protocol

A DVS event is represented as an input spike addressed to pixel \((x,y)\), with index \(i=yW+x\). Both polarities are treated alike in the current ESCM. Here every synthetic input timestamp falls exactly on a simulation tick, avoiding ambiguity from input rounding.

Stimuli are applied at 1 ms and later, then **no further input** occurs until 60 ms. All four scenarios use the same 5 × 5 connectivity and parameters. We interpret persistence as *continued output spiking*, not merely \(P=1\) or a large membrane voltage.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from brian2 import *

width = height = 5
N = width * height
center = 2 * width + 2
neighbor = center + 1

tau = 0.2 * ms
vt, vr = 0.1, 0.0
beta, Wi, Wk = 0.5, 6.0, -3.0
k = 8
clock_step = 0.5 * ms
end_ms = 60.0

stimuli = {
    "single": (np.array([center]), np.array([1.0])),
    "repeated": (np.array([center, center, center]), np.array([1.0, 5.0, 9.0])),
    "sequential": (np.array([center, neighbor]), np.array([1.0, 5.0])),
    "simultaneous": (np.array([center, neighbor]), np.array([1.0, 1.0])),
}

print(f"Connected {width}x{height} network; center={center}, neighbor={neighbor}")
print(f"tau={tau}, dt={clock_step}, vt={vt}, beta={beta}, Wi={Wi}, Wk={Wk}, k={k}")

## 2. Run the unchanged ESCM mechanism

For each scenario, the network starts from zero state. It evolves according to

\[
\tau\dot v_i=-v_i+\mathrm{ExtIn}_i+P_i(\mathrm{Exc}_i+\mathrm{Inh}_i), \qquad
\tau\dot z_i=-z_i\quad (z=\mathrm{ExtIn},\mathrm{Exc},\mathrm{Inh}).
\]

Camera events **set** \(\mathrm{ExtIn}=\beta\). Recurrent spikes set \(\mathrm{Exc}_{\rm pre}=W_i\); postsynaptic spike counters are incremented before assigning \(\mathrm{Inh}_{\rm post}=P_{\rm post}W_k\,\mathrm{counter}_{\rm post}\). Both custom persistence events are retained. The neuron-local counter is reset on a neuron spike, matching the corrected regression baseline.

The experiment records all 25 neurons' internal variables and output spikes. It does not rely on `BrianHF` or recorded DVS files.

In [ ]:
def simulate(input_ids, input_times_ms):
    start_scope()
    defaultclock.dt = clock_step

    params = {"tau": tau, "vt": vt, "vr": vr, "beta": beta, "Wi": Wi, "Wk": Wk}
    camera = SpikeGeneratorGroup(N, input_ids, input_times_ms * ms)
    equations = """
    dv/dt = (-v + ExtIn + P*(Exc + Inh))/tau : 1
    dExtIn/dt = -ExtIn/tau : 1
    dExc/dt = -Exc/tau : 1
    dInh/dt = -Inh/tau : 1
    incoming_spikes : integer
    P : integer
    """
    neurons = NeuronGroup(
        N, equations, threshold="v > vt",
        reset="v = vr; incoming_spikes = 0",
        refractory=0 * ms,
        events={"P_ON": "v > vt",
                "P_OFF": "(timestep(t - lastspike, dt) > timestep(dt, dt) and v <= vt)"},
        method="exact", namespace=params,
    )
    neurons.set_event_schedule("P_ON", when="after_thresholds")
    neurons.run_on_event("P_ON", "P = 1", when="after_thresholds")
    neurons.set_event_schedule("P_OFF", when="groups")
    neurons.run_on_event("P_OFF", "P = 0", when="groups")

    feed = Synapses(camera, neurons, on_pre="ExtIn_post = beta", namespace=params)
    feed.connect(j="i")
    recurrent = Synapses(
        neurons, neurons,
        on_pre={"pre": "incoming_spikes_post += 1; Exc_pre = Wi",
                "pre_2": "Inh_post = P_post * Wk * incoming_spikes_post"},
        namespace=params,
    )
    recurrent.pre.order = 0
    recurrent.pre_2.order = 1

    pre = np.repeat(np.arange(N), N)
    post = np.tile(np.arange(N), N)
    neighbors = (
        (pre != post)
        & (np.abs(pre % width - post % width) <= k)
        & (np.abs(pre // width - post // width) <= k)
    )
    recurrent.connect(i=pre[neighbors], j=post[neighbors])

    spikes = SpikeMonitor(neurons)
    states = StateMonitor(
        neurons, ["v", "P", "ExtIn", "Exc", "Inh", "incoming_spikes"],
        record=True, dt=clock_step,
    )
    net = Network(camera, neurons, feed, recurrent, spikes, states)
    net.run(end_ms * ms)

    return {
        "t": np.asarray(states.t / ms),
        "state": {name: np.asarray(getattr(states, name)).copy()
                  for name in ("v", "P", "ExtIn", "Exc", "Inh", "incoming_spikes")},
        "spike_t": np.asarray(spikes.t / ms),
        "spike_i": np.asarray(spikes.i, dtype=int),
        "input_t": np.asarray(input_times_ms),
        "input_i": np.asarray(input_ids),
    }

results = {}
for name, (ids, times) in stimuli.items():
    results[name] = simulate(ids, times)
    r = results[name]
    print(f"{name:13s}  input={len(ids):2d}  output={len(r['spike_i']):4d}")

## 3. What does the network actually output?

Compare all four stimulus protocols using output spike rasters. A vertical dashed line marks the **last input**, after which only recurrent dynamics can produce output.

The key comparisons are *single versus repeated* (does fresh input extend activity?) and *sequential versus simultaneous* (does neighborhood inhibition preferentially remove an older active location?).

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True, constrained_layout=True)
for ax, (name, r) in zip(axes, results.items()):
    ax.scatter(r["spike_t"], r["spike_i"], s=16, label="ESCM spikes")
    ax.scatter(r["input_t"], r["input_i"], s=65, marker="x",
               color="tab:red", linewidth=1.8, label="Camera input")
    ax.axvline(r["input_t"].max(), color="black", linestyle="--",
               linewidth=1, label="Last input")
    ax.set(title=name.capitalize(), ylabel="Neuron index",
           ylim=(-0.5, N - 0.5), yticks=np.arange(0, N, 5))
axes[0].legend(ncol=3, loc="upper right")
axes[-1].set(xlabel="Model time (ms)", xlim=(0, end_ms))
plt.show()

## 4. Quantify persistence and suppression

For each stimulated pixel, measure the first output spike latency, last output time after that pixel's last input, and output spikes during the **10 ms after the entire stimulus has ended**. These are descriptive diagnostics: the last spike is not necessarily a permanently sustained state, and a short firing sequence cannot demonstrate indefinite persistence.

We additionally compare the old center pixel and newly stimulated neighbor in the sequential case. If the new location suppresses the old one, the old location should become less active *after the second input*, while the new location remains responsive.

In [ ]:
print(f"{'Scenario':13s} {'Cell':>5s} {'Input end':>11s} {'First spike':>12s} {'Last spike':>11s} {'Post-input 10ms':>16s}")
print("-" * 78)
for name, r in results.items():
    all_end = r["input_t"].max()
    for cell in np.unique(r["input_i"]):
        input_start = r["input_t"][r["input_i"] == cell].min()
        input_end = r["input_t"][r["input_i"] == cell].max()
        out = r["spike_t"][r["spike_i"] == cell]
        response = out[out >= input_start]
        first = float(response.min()) if len(response) else np.nan
        last = float(response.max()) if len(response) else np.nan
        post = np.count_nonzero((out > all_end) & (out <= all_end + 10))
        print(f"{name:13s} {cell:5d} {input_end:11.1f} {first:12.1f} {last:11.1f} {post:16d}")

r = results["sequential"]
print("\nSequential case: activity after neighbor input at 5 ms")
for cell, label in [(center, "Original center"), (neighbor, "New neighbor")]:
    out = r["spike_t"][r["spike_i"] == cell]
    print(f"  {label:17s}: {np.count_nonzero((out >= 5) & (out < 15))} spikes from 5 to 15 ms")

## 5. Explain the observed dynamics

Show `v`, the persistence gate, self-excitatory trace, inhibitory trace, and the incoming-spike counter for the **center and neighbor**. These state variables can distinguish insufficient self-excitation from inhibition that terminates activity.

Note: Brian2's `StateMonitor` samples at its scheduled point in each timestep; discontinuous synaptic assignments may appear in the following sample. Do not infer sub-timestep ordering from the plot alone.

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(13, 12), sharex=True, constrained_layout=True)
for col, name in enumerate(("single", "sequential")):
    r = results[name]
    for row, key in enumerate(("v", "P", "Exc", "Inh", "incoming_spikes")):
        ax = axes[row, col]
        ax.plot(r["t"], r["state"][key][center], label="Center")
        ax.plot(r["t"], r["state"][key][neighbor], label="Neighbor", alpha=0.8)
        for onset in sorted(set(r["input_t"])):
            ax.axvline(onset, color="0.7", linestyle=":", linewidth=1)
        ax.set_ylabel(key)
        ax.grid(alpha=0.2)
        if row == 0:
            ax.set_title(name.capitalize())
            ax.axhline(vt, color="black", linestyle="--", linewidth=1, label="Threshold")
        if row == 4:
            ax.set_xlabel("Model time (ms)")
axes[0, 0].legend(ncol=3)
plt.show()

## Interpretation and next work package

The code above answers the baseline questions for one parameter set and timestep. Interpret the actual traces before redesigning:

- If the center keeps spiking with no more input, the existing recurrent feedback sustains activity.
- If it stops quickly, inspect whether `P` switches off, whether `Exc` decays between spikes, and whether incoming inhibition drives `v` down.
- If the second neighbor activates but does not selectively suppress the old neuron, quantify the current inhibition rule before replacing it.

Next we should perform **controlled ablations** (self-excitation, inhibition, and gate individually), then try a synaptic-current alternative. Any new mechanism must be tested on these same stimuli and eventually the real camera recordings.

**Execution status:** This notebook has been syntax-reviewed but not executed here; Brian2 is unavailable in this environment. Its output should be treated as the first empirical baseline only after you run it.